# Experiment: Fixed Scan-Budget Grid for MCMC-IS

Objective:
- At a fixed total MCMC-IS budget of 10 million evaluations, compare several fixed beta-selection budgets.
- For each scan budget, choose beta by the same local scan objective and then spend the remaining budget on production.
- Avoid duplicate production work by grouping identical selected beta/proposal pairs and evaluating all required production cutoffs from one cumulative run.

This is an offline calibration tool. After the run, we can fit a coarse rule of thumb such as scan budget as a function of total budget and/or \(p_0\).

In [1]:
from __future__ import annotations

from dataclasses import replace
import json
import os
import sys
from pathlib import Path

import pandas as pd
from IPython.display import Image, display


def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "perm_pval").exists() and (candidate / "results").exists():
            return candidate
    raise RuntimeError("Could not locate project root containing perm_pval/ and results/.")


project_root = find_project_root()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
os.environ.setdefault("MPLCONFIGDIR", str(project_root / ".matplotlib"))

from perm_pval.experiments.notebook_studies import (
    BetaSweepStudyConfig,
    CrossMethodStudyConfig,
    DEFAULT_MCMC_OBJECTIVE_GRID_Q_MULTIPLIERS,
    DEFAULT_MCMC_OBJECTIVE_GRID_SWAP_COUNTS,
    MCMCWorkflowConfig,
    MCMC_OBJECTIVE_GRID_REALISTIC_OBJECTIVES,
    SAMCWorkflowConfig,
    build_beta_workflow,
    create_timestamped_run_dir,
    load_beta_sweep_saved_output,
    load_cross_method_saved_output,
    load_mcmc_objective_grid_saved_output,
    load_selected_scenarios,
    run_mcmc_objective_grid_study,
    save_mcmc_objective_grid_outputs,
    regenerate_beta_sweep_plots_from_saved,
    regenerate_cross_method_plots_from_saved,
    run_beta_checkpoint_study,
    run_cross_method_study,
    save_beta_sweep_outputs,
    save_cross_method_outputs,
    summarize_records,
)

pd.set_option("display.max_columns", 100)
project_root

Matplotlib is building the font cache; this may take a moment.


PosixPath('/Users/noamchowers/Documents/University/Thesis/Code/MCMCIS')

In [2]:
import concurrent.futures as cf
import math
import time

import matplotlib.pyplot as plt
import numpy as np

from perm_pval.core.proposals import resolve_n_swap_pairs
from perm_pval.diagnostics.is_weights import effective_sample_size, summarize_weights
from perm_pval.experiments.notebook_studies import (
    _annotate_error_fields,
    _burn_in,
    _effective_n_jobs,
    _kept_samples_per_chain,
    _mcmc_eval_count,
    _mcmc_prefix_row,
    _run_single_chain_full_trace,
    _steps_per_chain,
    _try_make_process_pool,
    local_beta_scan,
    run_scan_budget_repeat_job,
)
from perm_pval.methods.beta_tuning import estimate_scale_T, iid_pilot_statistics

## Configuration

`TOTAL_BUDGET` is the full MCMC-IS evaluation budget: IID pilot + beta scan + production. `SCAN_BUDGETS` below are target beta-selection budgets, including the IID pilot. The actual charged budget can differ by a few evaluations because MCMC chains are split across chains and candidates.

The scan-budget grid is deliberately simple: it asks whether spending roughly `50k`, `100k`, `500k`, `1m`, or `2m` on beta selection is worthwhile when the full budget is `10m`.

In [3]:
FAST_MODE = False
SAVE_OUTPUTS = True

CATALOG_PATH = project_root / "results" / "exact_scenarios" / "v1" / "catalog.json"
OUTPUT_ROOT = project_root / "results" / "mcmcis_scan_budget_grid"

SCENARIO_KEYS_OVERRIDE = [
    "gwas_additive_score_n40",
    "linear_stat_dp_n40",
    "rank_sum_dp_n40",
]
MIN_TAIL_STATES = 2

TOTAL_BUDGET = 10_000_000 if not FAST_MODE else 200_000
N_REPEATS = 6 if not FAST_MODE else 1
N_JOBS = min(6, os.cpu_count() or 1) if not FAST_MODE else 1
BASE_SEED = 83_083

SCAN_BUDGETS = (
    50_000,
    100_000,
    500_000,
    1_000_000,
    2_000_000,
) if not FAST_MODE else (25_000, 50_000)
SCAN_FINALIST_COUNT = 4
SCAN_FINAL_TO_SCREEN_RATIO = 2.0

Q_MULTIPLIERS = (0.001, 0.005, 0.01, 0.03, 0.05, 0.075, 0.10, 0.15, 0.20, 0.25, 0.33, 1.0)
MCMC_LOCAL_SCAN_OBJECTIVE = "varhat_qmatch_soft"
PRODUCTION_ESTIMATE_VARIANCE = False
MCMC_PROPOSAL_SIZE_BY_SAMPLE_BAND = {
    "small": 1,
    "medium": 1,
    "large": 2,
}

BUDGET_RULE_SPECS = [
    {
        "name": f"scan_{budget // 1_000_000}m" if budget >= 1_000_000 else f"scan_{budget // 1_000}k",
        "kind": "fixed_scan_budget",
        "target_beta_selection_budget": int(budget),
        "finalist_count": SCAN_FINALIST_COUNT,
        "final_to_screen_ratio": SCAN_FINAL_TO_SCREEN_RATIO,
    }
    for budget in SCAN_BUDGETS
]

base_mcmc_cfg = MCMCWorkflowConfig(
    pilot_samples=25_000 if not FAST_MODE else 5_000,
    chains=2,
    thin=1,
    estimate_variance=True,
    proposal_size=1,
    local_scan_objective=MCMC_LOCAL_SCAN_OBJECTIVE,
    local_scan_q_multipliers=Q_MULTIPLIERS,
    local_scan_screen_chains=1,
    local_scan_chains=2,
    local_scan_burn_in_fraction=0.20,
    local_scan_thin=1,
)

print(json.dumps({
    "FAST_MODE": FAST_MODE,
    "SCENARIO_KEYS_OVERRIDE": SCENARIO_KEYS_OVERRIDE,
    "TOTAL_BUDGET": TOTAL_BUDGET,
    "N_REPEATS": N_REPEATS,
    "N_JOBS": N_JOBS,
    "SCAN_BUDGETS": SCAN_BUDGETS,
    "SCAN_FINALIST_COUNT": SCAN_FINALIST_COUNT,
    "SCAN_FINAL_TO_SCREEN_RATIO": SCAN_FINAL_TO_SCREEN_RATIO,
    "Q_MULTIPLIERS": Q_MULTIPLIERS,
    "MCMC_LOCAL_SCAN_OBJECTIVE": MCMC_LOCAL_SCAN_OBJECTIVE,
    "PRODUCTION_ESTIMATE_VARIANCE": PRODUCTION_ESTIMATE_VARIANCE,
    "MCMC_PROPOSAL_SIZE_BY_SAMPLE_BAND": MCMC_PROPOSAL_SIZE_BY_SAMPLE_BAND,
    "BUDGET_RULE_SPECS": BUDGET_RULE_SPECS,
}, indent=2))

{
  "FAST_MODE": false,
  "SCENARIO_KEYS_OVERRIDE": [
    "gwas_additive_score_n40",
    "linear_stat_dp_n40",
    "rank_sum_dp_n40"
  ],
  "TOTAL_BUDGET": 10000000,
  "N_REPEATS": 6,
  "N_JOBS": 6,
  "SCAN_BUDGETS": [
    50000,
    100000,
    500000,
    1000000,
    2000000
  ],
  "SCAN_FINALIST_COUNT": 4,
  "SCAN_FINAL_TO_SCREEN_RATIO": 2.0,
  "Q_MULTIPLIERS": [
    0.001,
    0.005,
    0.01,
    0.03,
    0.05,
    0.075,
    0.1,
    0.15,
    0.2,
    0.25,
    0.33,
    1.0
  ],
  "MCMC_LOCAL_SCAN_OBJECTIVE": "varhat_qmatch_soft",
  "PRODUCTION_ESTIMATE_VARIANCE": false,
  "MCMC_PROPOSAL_SIZE_BY_SAMPLE_BAND": {
    "small": 1,
    "medium": 1,
    "large": 2
  },
  "BUDGET_RULE_SPECS": [
    {
      "name": "scan_50k",
      "kind": "fixed_scan_budget",
      "target_beta_selection_budget": 50000,
      "finalist_count": 4,
      "final_to_screen_ratio": 2.0
    },
    {
      "name": "scan_100k",
      "kind": "fixed_scan_budget",
      "target_beta_selection_budget": 100000

## Helpers

In [4]:
def mcmc_proposal_size_for_scenario(scenario) -> int:
    band = str(scenario.portfolio.get("sample_size_band", "medium"))
    return int(MCMC_PROPOSAL_SIZE_BY_SAMPLE_BAND.get(band, 1))


def q_target_from_p0(p0: float, cfg: MCMCWorkflowConfig) -> float:
    return float(float(p0) ** float(cfg.d_alpha))


def kept_samples_total(total_steps: int, *, n_chains: int, burn_in_fraction: float, thin: int) -> int:
    steps_per_chain = _steps_per_chain(int(total_steps), int(n_chains))
    burn_in = _burn_in(steps_per_chain, float(burn_in_fraction))
    return int(n_chains) * _kept_samples_per_chain(steps_per_chain, burn_in, int(thin))


def stage_eval_total(total_steps: int, *, n_candidates: int, n_chains: int) -> int:
    steps_per_chain = _steps_per_chain(int(total_steps), int(n_chains))
    return int(n_candidates) * _mcmc_eval_count(steps_per_chain, int(n_chains))


def split_fixed_scan_budget(
    *,
    target_beta_selection_budget: int,
    cfg: MCMCWorkflowConfig,
    n_candidates: int,
    finalist_count: int,
    final_to_screen_ratio: float,
) -> tuple[int, int]:
    scan_budget_ex_pilot = max(int(target_beta_selection_budget) - int(cfg.pilot_samples), 1)
    budget_units = float(n_candidates) + float(finalist_count) * float(final_to_screen_ratio)
    screen_total_steps = max(int(scan_budget_ex_pilot / max(budget_units, 1.0)), 1)
    final_total_steps = max(int(round(float(final_to_screen_ratio) * screen_total_steps)), 1)
    return int(screen_total_steps), int(final_total_steps)


def resolve_budget_rule(rule: dict, scenario, cfg: MCMCWorkflowConfig) -> dict:
    proposal_size = int(mcmc_proposal_size_for_scenario(scenario))
    n_candidates = len(tuple(cfg.local_scan_q_multipliers))
    finalist_count = min(int(rule.get("finalist_count", cfg.local_scan_finalist_count)), n_candidates)
    q_target = q_target_from_p0(scenario.exact_p, cfg)

    if rule["kind"] != "fixed_scan_budget":
        raise ValueError(f"Unknown budget rule kind for this notebook: {rule['kind']}")

    target_beta_selection_budget = int(rule["target_beta_selection_budget"])
    if target_beta_selection_budget >= int(TOTAL_BUDGET):
        raise ValueError("target_beta_selection_budget must be smaller than TOTAL_BUDGET")
    final_to_screen_ratio = float(rule.get("final_to_screen_ratio", 2.0))
    screen_total_steps, final_total_steps = split_fixed_scan_budget(
        target_beta_selection_budget=target_beta_selection_budget,
        cfg=cfg,
        n_candidates=n_candidates,
        finalist_count=finalist_count,
        final_to_screen_ratio=final_to_screen_ratio,
    )

    screen_eval_total = stage_eval_total(
        screen_total_steps,
        n_candidates=n_candidates,
        n_chains=int(cfg.local_scan_screen_chains),
    )
    final_eval_total_upper = stage_eval_total(
        final_total_steps,
        n_candidates=finalist_count,
        n_chains=int(cfg.local_scan_chains),
    )
    expected_beta_selection_budget = int(cfg.pilot_samples) + int(screen_eval_total) + int(final_eval_total_upper)
    return {
        "rule_name": str(rule["name"]),
        "rule_kind": str(rule["kind"]),
        "proposal_size": proposal_size,
        "n_candidates": int(n_candidates),
        "finalist_count": int(finalist_count),
        "q_target": float(q_target),
        "q_ref": np.nan,
        "tail_hit_steps": np.nan,
        "screen_cap_steps": np.nan,
        "target_beta_selection_budget": int(target_beta_selection_budget),
        "target_scan_budget_ex_pilot": int(max(target_beta_selection_budget - int(cfg.pilot_samples), 1)),
        "final_to_screen_ratio": float(final_to_screen_ratio),
        "screen_total_steps": int(screen_total_steps),
        "final_total_steps": int(final_total_steps),
        "expected_screen_eval_total": int(screen_eval_total),
        "expected_final_eval_total_upper": int(final_eval_total_upper),
        "expected_beta_selection_budget_upper": int(expected_beta_selection_budget),
    }


def mcmc_cfg_for_budget_rule(rule_budget: dict, cfg: MCMCWorkflowConfig) -> MCMCWorkflowConfig:
    return replace(
        cfg,
        proposal_size=int(rule_budget["proposal_size"]),
        local_scan_swap_counts=(int(rule_budget["proposal_size"]),),
        local_scan_finalist_count=int(rule_budget["finalist_count"]),
        local_scan_screen_total_steps=int(rule_budget["screen_total_steps"]),
        local_scan_total_steps=int(rule_budget["final_total_steps"]),
    )


def selected_scan_row(scan: dict) -> dict:
    rows = list(scan.get("rows", []))
    if not rows:
        return {}
    beta = float(scan.get("selected_beta", np.nan))
    prop = int(scan.get("selected_proposal_size", -1))
    matches = [
        row
        for row in rows
        if int(row.get("n_swap_pairs", -999)) == prop
        and np.isfinite(row.get("beta", np.nan))
        and abs(float(row.get("beta")) - beta) <= 1e-10
    ]
    return dict(matches[0] if matches else rows[0])

## Load Scenarios

In [5]:
scenarios = load_selected_scenarios(
    catalog_path=CATALOG_PATH,
    scenario_keys=SCENARIO_KEYS_OVERRIDE,
    portfolio_group=None,
    min_tail_states=MIN_TAIL_STATES,
)
scenario_by_key = {scenario.key: scenario for scenario in scenarios}
run_dir = create_timestamped_run_dir(OUTPUT_ROOT, "scan_budget_grid") if SAVE_OUTPUTS else None

pd.DataFrame([
    {
        "scenario": s.key,
        "exact_p": s.exact_p,
        "family": s.portfolio.get("family"),
        "statistic_family": s.portfolio.get("statistic_family"),
        "sample_size_band": s.portfolio.get("sample_size_band"),
        "proposal_size": mcmc_proposal_size_for_scenario(s),
    }
    for s in scenarios
])

,scenario,exact_p,family,statistic_family,sample_size_band,proposal_size
0,gwas_additive_score_n40,9.121811e-07,gwas_additive_score,linear_statistic,medium,1
1,linear_stat_dp_n40,8.124978e-09,linear_stat_dp,linear_statistic,medium,1
2,rank_sum_dp_n40,8.445624e-08,rank_sum_dp,rank_based,medium,1


## Run Fixed-Budget Scans

Each `(scenario, repeat)` job uses one common IID pilot and reuses it across all scan budgets. Jobs are independent, so the notebook can parallelize at that outer level while still charging the pilot cost to every budget setting.

In [6]:
scan_records: list[dict] = []
production_records: list[dict] = []
repeat_jobs: list[dict] = []

for scenario_idx, scenario in enumerate(scenarios):
    rule_budgets = [resolve_budget_rule(rule, scenario, base_mcmc_cfg) for rule in BUDGET_RULE_SPECS]
    print(
        f"Prepared scenario {scenario.key} | exact p={scenario.exact_p:.3e} "
        f"| repeats={N_REPEATS} | rules={len(rule_budgets)}"
    )
    for repeat_idx in range(N_REPEATS):
        pilot_seed = BASE_SEED + 100_000 * scenario_idx + 10_000 * repeat_idx
        repeat_jobs.append(
            {
                "scenario_key": str(scenario.key),
                "scenario_display": str(scenario.description),
                "family": scenario.portfolio.get("family"),
                "statistic_family": scenario.portfolio.get("statistic_family"),
                "sample_size_band": scenario.portfolio.get("sample_size_band"),
                "problem": scenario.problem,
                "exact_p": float(scenario.exact_p),
                "repeat_idx": int(repeat_idx),
                "pilot_seed": int(pilot_seed),
                "total_budget": int(TOTAL_BUDGET),
                "base_mcmc_cfg": base_mcmc_cfg,
                "rule_budgets": [dict(rule_budget) for rule_budget in rule_budgets],
                "production_estimate_variance": bool(PRODUCTION_ESTIMATE_VARIANCE),
            }
        )

t0 = time.perf_counter()
n_workers = _effective_n_jobs(N_JOBS, len(repeat_jobs))
executor = _try_make_process_pool(n_workers) if n_workers > 1 else None
print(f"Running {len(repeat_jobs)} scenario-repeat jobs with n_workers={n_workers}")

def _consume_job_result(result: dict) -> None:
    scan_records.extend(result["scan_records"])
    production_records.extend(result["production_records"])


if executor is None:
    for job_idx, job in enumerate(repeat_jobs, start=1):
        _consume_job_result(run_scan_budget_repeat_job(**job))
        if job_idx % max(1, len(repeat_jobs) // 6) == 0 or job_idx == len(repeat_jobs):
            print(f"Completed {job_idx}/{len(repeat_jobs)} jobs")
else:
    with executor:
        futures = [executor.submit(run_scan_budget_repeat_job, **job) for job in repeat_jobs]
        for job_idx, future in enumerate(cf.as_completed(futures), start=1):
            _consume_job_result(future.result())
            if job_idx % max(1, len(repeat_jobs) // 6) == 0 or job_idx == len(repeat_jobs):
                print(f"Completed {job_idx}/{len(repeat_jobs)} jobs")

scan_df = pd.DataFrame(scan_records)
production_df = pd.DataFrame(production_records)
print(
    f"Completed {len(scan_df)} scans and {len(production_df)} production rows "
    f"in {time.perf_counter() - t0:.1f}s"
)
display(scan_df.sort_values(["scenario", "repeat", "rule_name"])[[
    "scenario", "repeat", "rule_name", "beta_selection_budget", "production_budget",
    "screen_total_steps", "final_total_steps", "scan_n_weighted_samples", "selected_q_multiplier",
    "selected_beta", "selected_q_tilt_tail_share", "selected_ess", "selected_weight_cv",
]])

Prepared scenario gwas_additive_score_n40 | exact p=9.122e-07 | repeats=6 | rules=5
Prepared scenario linear_stat_dp_n40 | exact p=8.125e-09 | repeats=6 | rules=5
Prepared scenario rank_sum_dp_n40 | exact p=8.446e-08 | repeats=6 | rules=5
Running 18 scenario-repeat jobs with n_workers=6
Completed 3/18 jobs
Completed 6/18 jobs
Completed 9/18 jobs
Completed 12/18 jobs
Completed 15/18 jobs
Completed 18/18 jobs
Completed 90 scans and 180 production rows in 5853.2s


,scenario,repeat,rule_name,beta_selection_budget,production_budget,screen_total_steps,final_total_steps,scan_n_weighted_samples,selected_q_multiplier,selected_beta,selected_q_tilt_tail_share,selected_ess,selected_weight_cv
21,gwas_additive_score_n40,0,scan_100k,100020,9899980,3750,7500,60000,1.000,3.544922,0.012000,34.074296,13.231999
23,gwas_additive_score_n40,0,scan_1m,1000020,8999980,48750,97500,780000,0.075,2.301758,0.001692,600.406682,11.353940
24,gwas_additive_score_n40,0,scan_2m,2000020,7999980,98750,197500,1580000,0.075,2.301758,0.001633,1282.911608,11.052482
22,gwas_additive_score_n40,0,scan_500k,500020,9499980,23750,47500,380000,0.200,2.726562,0.004684,90.757787,20.437633
20,gwas_additive_score_n40,0,scan_50k,47518,9952482,1250,2500,18000,0.330,2.963867,0.000500,57.069241,5.834822
...,...,...,...,...,...,...,...,...,...,...,...,...,...
81,rank_sum_dp_n40,5,scan_100k,100020,9899980,3750,7500,60000,0.075,2.879883,0.000833,108.295172,7.375916
83,rank_sum_dp_n40,5,scan_1m,1000020,8999980,48750,97500,780000,0.100,3.029297,0.001256,290.355120,16.359601
84,rank_sum_dp_n40,5,scan_2m,2000020,7999980,98750,197500,1580000,0.050,2.684570,0.000829,1065.869096,12.134078
82,rank_sum_dp_n40,5,scan_500k,500020,9499980,23750,47500,380000,0.050,2.684570,0.000632,474.186292,8.895914


## Run Deduplicated Production

Production is already executed inside each parallel `(scenario, repeat)` job. Within a job, if multiple scan budgets select the same beta/proposal, we run one cumulative production chain up to the largest required leftover budget and read off the shorter checkpoints for the other scan budgets.

We intentionally do not reuse scan final states here. That keeps the production runs groupable and isolates the beta/proposal selection effect from differences in scan-generated initialization states.

In [7]:
print(f"Loaded {len(production_df)} policy production rows from the completed scenario-repeat jobs")
display(production_df.sort_values(["scenario", "repeat", "rule_name", "estimator_variant"])[[
    "scenario", "repeat", "rule_name", "estimator_variant", "estimate", "exact_p", "root_squared_error",
    "abs_log10_error", "selected_q_multiplier", "selected_beta", "beta_selection_budget",
    "production_budget", "available_scan_n_weighted_samples", "scan_n_weighted_samples",
    "production_n_weighted_samples", "ess", "weight_cv", "acceptance_rate",
]])

Loaded 180 policy production rows from the completed scenario-repeat jobs


,scenario,repeat,rule_name,estimator_variant,estimate,exact_p,root_squared_error,abs_log10_error,selected_q_multiplier,selected_beta,beta_selection_budget,production_budget,available_scan_n_weighted_samples,scan_n_weighted_samples,production_n_weighted_samples,ess,weight_cv,acceptance_rate
48,gwas_additive_score_n40,0,scan_100k,production_only,9.742447e-07,9.121811e-07,6.206359e-08,0.028587,1.000,3.544922,100020,9899980,60000,0,7919984,1633.729155,69.618927,0.525472
49,gwas_additive_score_n40,0,scan_100k,scan_plus_production,9.747336e-07,9.121811e-07,6.255250e-08,0.028805,1.000,3.544922,100020,9899980,60000,60000,7919984,1639.425349,69.760659,0.525472
40,gwas_additive_score_n40,0,scan_1m,production_only,9.392205e-07,9.121811e-07,2.703937e-08,0.012686,0.075,2.301758,1000020,8999980,780000,0,7199984,29886.427247,15.489077,0.662052
41,gwas_additive_score_n40,0,scan_1m,scan_plus_production,9.604797e-07,9.121811e-07,4.829858e-08,0.022407,0.075,2.301758,1000020,8999980,780000,780000,7199984,27714.388371,16.939200,0.662052
42,gwas_additive_score_n40,0,scan_2m,production_only,9.371508e-07,9.121811e-07,2.496974e-08,0.011728,0.075,2.301758,2000020,7999980,1580000,0,6399984,24990.709338,15.971679,0.662143
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
163,rank_sum_dp_n40,5,scan_2m,scan_plus_production,9.980132e-08,8.445624e-08,1.534507e-08,0.072505,0.050,2.684570,2000020,7999980,1580000,1580000,6399984,321.007543,157.664768,0.550233
160,rank_sum_dp_n40,5,scan_500k,production_only,8.414510e-08,8.445624e-08,3.111432e-10,0.001603,0.050,2.684570,500020,9499980,380000,0,7599984,19943.448589,19.495556,0.550349
161,rank_sum_dp_n40,5,scan_500k,scan_plus_production,6.184621e-08,8.445624e-08,2.261004e-08,0.135319,0.050,2.684570,500020,9499980,380000,380000,7599984,34.418147,481.511130,0.550349
168,rank_sum_dp_n40,5,scan_50k,production_only,9.304763e-08,8.445624e-08,8.591382e-09,0.042074,0.250,3.605469,50020,9949980,20000,0,7959984,2508.696597,56.320121,0.437200


## Summarize Fixed Scan Budgets

In [8]:
def summarize_policy_records(df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for (scenario, rule_name, estimator_variant), sub in df.groupby(["scenario", "rule_name", "estimator_variant"], sort=True):
        exact_p = float(sub["exact_p"].iloc[0])
        estimates = sub["estimate"].astype(float).to_numpy()
        rows.append({
            "scenario": scenario,
            "rule_name": rule_name,
            "estimator_variant": estimator_variant,
            "n_runs": int(len(sub)),
            "exact_p": exact_p,
            "mean_estimate": float(np.mean(estimates)),
            "rmse": float(np.sqrt(np.mean((estimates - exact_p) ** 2))),
            "mean_abs_log10_error": float(np.nanmean(sub["abs_log10_error"].astype(float))),
            "mean_selected_q_multiplier": float(np.nanmean(sub["selected_q_multiplier"].astype(float))),
            "median_selected_q_multiplier": float(np.nanmedian(sub["selected_q_multiplier"].astype(float))),
            "mean_beta_selection_budget": float(np.mean(sub["beta_selection_budget"].astype(float))),
            "mean_production_budget": float(np.mean(sub["production_budget"].astype(float))),
            "mean_q_tilt_tail_share": float(np.nanmean(sub["tail_share_raw"].astype(float))),
            "mean_ess": float(np.nanmean(sub["ess"].astype(float))),
            "mean_weight_cv": float(np.nanmean(sub["weight_cv"].astype(float))),
            "mean_acceptance_rate": float(np.nanmean(sub["acceptance_rate"].astype(float))),
            "mean_available_scan_n_weighted_samples": float(np.nanmean(sub["available_scan_n_weighted_samples"].astype(float))),
            "mean_scan_n_weighted_samples": float(np.nanmean(sub["scan_n_weighted_samples"].astype(float))),
            "mean_production_n_weighted_samples": float(np.nanmean(sub["production_n_weighted_samples"].astype(float))),
        })
    return pd.DataFrame(rows).sort_values(["scenario", "estimator_variant", "rmse", "mean_abs_log10_error"]).reset_index(drop=True)


policy_summary = summarize_policy_records(production_df)
policy_summary["rmse_rank"] = policy_summary.groupby(["scenario", "estimator_variant"])["rmse"].rank(method="min")
policy_summary["abs_log10_error_rank"] = policy_summary.groupby(["scenario", "estimator_variant"])["mean_abs_log10_error"].rank(method="min")
display(policy_summary)

leaderboard = (
    policy_summary
    .groupby(["estimator_variant", "rule_name"], as_index=False)
    .agg(
        mean_rmse_rank=("rmse_rank", "mean"),
        mean_abs_log10_error_rank=("abs_log10_error_rank", "mean"),
        mean_rmse=("rmse", "mean"),
        mean_abs_log10_error=("mean_abs_log10_error", "mean"),
        mean_beta_selection_budget=("mean_beta_selection_budget", "mean"),
        mean_production_budget=("mean_production_budget", "mean"),
        mean_available_scan_n_weighted_samples=("mean_available_scan_n_weighted_samples", "mean"),
        mean_scan_n_weighted_samples=("mean_scan_n_weighted_samples", "mean"),
        mean_production_n_weighted_samples=("mean_production_n_weighted_samples", "mean"),
        mean_ess=("mean_ess", "mean"),
        mean_weight_cv=("mean_weight_cv", "mean"),
    )
    .sort_values(["estimator_variant", "mean_rmse_rank", "mean_abs_log10_error_rank", "mean_rmse"])
    .reset_index(drop=True)
)
display(leaderboard)

comparison_rows = []
for (scenario, rule_name), sub in policy_summary.groupby(["scenario", "rule_name"], sort=True):
    if set(sub["estimator_variant"]) != {"production_only", "scan_plus_production"}:
        continue
    base = sub[sub["estimator_variant"] == "production_only"].iloc[0]
    pooled = sub[sub["estimator_variant"] == "scan_plus_production"].iloc[0]
    comparison_rows.append({
        "scenario": scenario,
        "rule_name": rule_name,
        "exact_p": float(base["exact_p"]),
        "production_only_rmse": float(base["rmse"]),
        "scan_plus_production_rmse": float(pooled["rmse"]),
        "rmse_ratio_scan_over_prod": (
            float(pooled["rmse"] / base["rmse"])
            if np.isfinite(base["rmse"]) and float(base["rmse"]) > 0.0
            else np.nan
        ),
        "production_only_abs_log10_error": float(base["mean_abs_log10_error"]),
        "scan_plus_production_abs_log10_error": float(pooled["mean_abs_log10_error"]),
        "abs_log10_error_delta": float(pooled["mean_abs_log10_error"] - base["mean_abs_log10_error"]),
        "scan_n_weighted_samples_added": float(pooled["mean_scan_n_weighted_samples"]),
    })
comparison_df = pd.DataFrame(comparison_rows).sort_values(["scenario", "rmse_ratio_scan_over_prod"])
display(comparison_df)

/var/folders/41/087q_7ns2kvc9g1r9zyhz2pw0000gp/T/ipykernel_7280/1272690404.py:19: RuntimeWarning: Mean of empty slice
  "mean_q_tilt_tail_share": float(np.nanmean(sub["tail_share_raw"].astype(float))),
/var/folders/41/087q_7ns2kvc9g1r9zyhz2pw0000gp/T/ipykernel_7280/1272690404.py:19: RuntimeWarning: Mean of empty slice
  "mean_q_tilt_tail_share": float(np.nanmean(sub["tail_share_raw"].astype(float))),
/var/folders/41/087q_7ns2kvc9g1r9zyhz2pw0000gp/T/ipykernel_7280/1272690404.py:19: RuntimeWarning: Mean of empty slice
  "mean_q_tilt_tail_share": float(np.nanmean(sub["tail_share_raw"].astype(float))),
/var/folders/41/087q_7ns2kvc9g1r9zyhz2pw0000gp/T/ipykernel_7280/1272690404.py:19: RuntimeWarning: Mean of empty slice
  "mean_q_tilt_tail_share": float(np.nanmean(sub["tail_share_raw"].astype(float))),
/var/folders/41/087q_7ns2kvc9g1r9zyhz2pw0000gp/T/ipykernel_7280/1272690404.py:19: RuntimeWarning: Mean of empty slice
  "mean_q_tilt_tail_share": float(np.nanmean(sub["tail_share_raw"].astype(

,scenario,rule_name,estimator_variant,n_runs,exact_p,mean_estimate,rmse,mean_abs_log10_error,mean_selected_q_multiplier,median_selected_q_multiplier,mean_beta_selection_budget,mean_production_budget,mean_q_tilt_tail_share,mean_ess,mean_weight_cv,mean_acceptance_rate,mean_available_scan_n_weighted_samples,mean_scan_n_weighted_samples,mean_production_n_weighted_samples,rmse_rank,abs_log10_error_rank
0,gwas_additive_score_n40,scan_1m,production_only,6,9.121811e-07,9.054146e-07,2.476354e-08,0.009837,0.150000,0.1750,1000020.0,8999980.0,0.004395,23731.868295,22.499395,0.628976,7.800000e+05,0.000000e+00,7.199984e+06,1.0,1.0
1,gwas_additive_score_n40,scan_100k,production_only,6,9.121811e-07,9.287915e-07,3.249904e-08,0.011364,0.297500,0.1750,100020.0,9899980.0,0.007972,49603.435675,26.567527,0.617708,6.000000e+04,0.000000e+00,7.919984e+06,2.0,2.0
2,gwas_additive_score_n40,scan_2m,production_only,6,9.121811e-07,9.390163e-07,3.514988e-08,0.012466,0.150000,0.1500,2000020.0,7999980.0,0.004380,18380.024133,23.422060,0.630364,1.580000e+06,0.000000e+00,6.399984e+06,3.0,3.0
3,gwas_additive_score_n40,scan_50k,production_only,6,9.121811e-07,8.970467e-07,4.783620e-08,0.018534,0.278333,0.3300,49186.0,9950814.0,0.007949,11212.077984,35.722906,0.595479,1.933333e+04,0.000000e+00,7.960651e+06,4.0,4.0
4,gwas_additive_score_n40,scan_500k,production_only,6,9.121811e-07,8.920788e-07,6.183380e-08,0.025356,0.185000,0.2250,500020.0,9499980.0,0.005341,125988.309305,29.746746,0.631324,3.800000e+05,0.000000e+00,7.599984e+06,5.0,5.0
5,gwas_additive_score_n40,scan_1m,scan_plus_production,6,9.121811e-07,9.191935e-07,3.213020e-08,0.012775,0.150000,0.1750,1000020.0,8999980.0,NaN,21089.561523,23.987351,0.628976,7.800000e+05,7.800000e+05,7.199984e+06,1.0,2.0
6,gwas_additive_score_n40,scan_100k,scan_plus_production,6,9.121811e-07,9.328292e-07,3.317817e-08,0.011933,0.297500,0.1750,100020.0,9899980.0,NaN,29479.985594,27.251838,0.617708,6.000000e+04,6.000000e+04,7.919984e+06,2.0,1.0
7,gwas_additive_score_n40,scan_2m,scan_plus_production,6,9.121811e-07,9.480704e-07,4.398762e-08,0.016607,0.150000,0.1500,2000020.0,7999980.0,NaN,9233.693934,32.024294,0.630364,1.580000e+06,1.580000e+06,6.399984e+06,3.0,3.0
8,gwas_additive_score_n40,scan_50k,scan_plus_production,6,9.121811e-07,8.982445e-07,4.653619e-08,0.017833,0.278333,0.3300,49186.0,9950814.0,NaN,11253.535236,35.715865,0.595479,1.933333e+04,1.933333e+04,7.960651e+06,4.0,4.0
9,gwas_additive_score_n40,scan_500k,scan_plus_production,6,9.121811e-07,9.165901e-07,5.338369e-08,0.021498,0.185000,0.2250,500020.0,9499980.0,NaN,9192.684637,38.505080,0.631324,3.800000e+05,3.800000e+05,7.599984e+06,5.0,5.0


,estimator_variant,rule_name,mean_rmse_rank,mean_abs_log10_error_rank,mean_rmse,mean_abs_log10_error,mean_beta_selection_budget,mean_production_budget,mean_available_scan_n_weighted_samples,mean_scan_n_weighted_samples,mean_production_n_weighted_samples,mean_ess,mean_weight_cv
0,production_only,scan_1m,1.666667,1.666667,1.010452e-08,0.018531,1000020.0,8999980.0,7.800000e+05,0.000000e+00,7.199984e+06,13096.974431,53.482436
1,production_only,scan_2m,2.000000,2.333333,1.280497e-08,0.019920,2000020.0,7999980.0,1.580000e+06,0.000000e+00,6.399984e+06,11495.407773,54.009907
2,production_only,scan_500k,3.333333,3.000000,2.189392e-08,0.033654,500020.0,9499980.0,3.800000e+05,0.000000e+00,7.599984e+06,48384.128708,57.434401
3,production_only,scan_100k,3.333333,3.333333,1.652889e-08,0.044385,100020.0,9899980.0,6.000000e+04,0.000000e+00,7.919984e+06,18589.185442,135.227849
4,production_only,scan_50k,4.666667,4.666667,2.597638e-08,0.111395,48491.0,9951509.0,1.877778e+04,0.000000e+00,7.961207e+06,4360.785214,172.346380
5,scan_plus_production,scan_1m,2.000000,2.333333,1.409996e-08,0.048232,1000020.0,8999980.0,7.800000e+05,7.800000e+05,7.199984e+06,8516.260978,67.956715
6,scan_plus_production,scan_500k,2.666667,2.666667,2.175734e-08,0.041872,500020.0,9499980.0,3.800000e+05,3.800000e+05,7.599984e+06,5407.234928,89.762070
7,scan_plus_production,scan_2m,2.666667,3.000000,1.922710e-08,0.048450,2000020.0,7999980.0,1.580000e+06,1.580000e+06,6.399984e+06,3938.494575,100.913045
8,scan_plus_production,scan_100k,3.000000,2.333333,1.676806e-08,0.046104,100020.0,9899980.0,6.000000e+04,6.000000e+04,7.919984e+06,11811.572768,135.692296
9,scan_plus_production,scan_50k,4.666667,4.666667,2.550764e-08,0.110820,48491.0,9951509.0,1.877778e+04,1.877778e+04,7.961207e+06,4334.870792,172.511788


,scenario,rule_name,exact_p,production_only_rmse,scan_plus_production_rmse,rmse_ratio_scan_over_prod,production_only_abs_log10_error,scan_plus_production_abs_log10_error,abs_log10_error_delta,scan_n_weighted_samples_added
3,gwas_additive_score_n40,scan_500k,9.121811e-07,6.183380e-08,5.338369e-08,0.863342,0.025356,0.021498,-0.003857,3.800000e+05
4,gwas_additive_score_n40,scan_50k,9.121811e-07,4.783620e-08,4.653619e-08,0.972824,0.018534,0.017833,-0.000701,1.933333e+04
0,gwas_additive_score_n40,scan_100k,9.121811e-07,3.249904e-08,3.317817e-08,1.020897,0.011364,0.011933,0.000569,6.000000e+04
2,gwas_additive_score_n40,scan_2m,9.121811e-07,3.514988e-08,4.398762e-08,1.251430,0.012466,0.016607,0.004141,1.580000e+06
1,gwas_additive_score_n40,scan_1m,9.121811e-07,2.476354e-08,3.213020e-08,1.297480,0.009837,0.012775,0.002939,7.800000e+05
8,linear_stat_dp_n40,scan_500k,8.124978e-09,1.345582e-09,1.312382e-09,0.975326,0.067103,0.063413,-0.003690,3.800000e+05
9,linear_stat_dp_n40,scan_50k,8.124978e-09,1.241985e-08,1.242846e-08,1.000694,0.235930,0.235793,-0.000137,1.800000e+04
5,linear_stat_dp_n40,scan_100k,8.124978e-09,2.954340e-09,2.991517e-09,1.012584,0.073787,0.078610,0.004823,6.000000e+04
7,linear_stat_dp_n40,scan_2m,8.124978e-09,8.127630e-10,2.267292e-09,2.789610,0.037077,0.081794,0.044717,1.580000e+06
6,linear_stat_dp_n40,scan_1m,8.124978e-09,4.717100e-10,4.046890e-09,8.579191,0.024106,0.108511,0.084404,7.800000e+05


## Save Outputs

In [9]:
if SAVE_OUTPUTS and run_dir is not None:
    run_dir.mkdir(parents=True, exist_ok=True)
    scan_df.to_json(run_dir / "scan_records.jsonl", orient="records", lines=True)
    production_df.to_json(run_dir / "production_records.jsonl", orient="records", lines=True)
    policy_summary.to_json(run_dir / "policy_summary.json", orient="records", indent=2)
    leaderboard.to_json(run_dir / "policy_leaderboard.json", orient="records", indent=2)
    comparison_df.to_json(run_dir / "estimator_comparison.json", orient="records", indent=2)
    (run_dir / "metadata.json").write_text(
        json.dumps(
            {
                "TOTAL_BUDGET": TOTAL_BUDGET,
                "N_REPEATS": N_REPEATS,
                "N_JOBS": N_JOBS,
                "SCENARIO_KEYS_OVERRIDE": SCENARIO_KEYS_OVERRIDE,
                "SCAN_BUDGETS": SCAN_BUDGETS,
                "SCAN_FINALIST_COUNT": SCAN_FINALIST_COUNT,
                "SCAN_FINAL_TO_SCREEN_RATIO": SCAN_FINAL_TO_SCREEN_RATIO,
                "Q_MULTIPLIERS": Q_MULTIPLIERS,
                "BUDGET_RULE_SPECS": BUDGET_RULE_SPECS,
                "MCMC_LOCAL_SCAN_OBJECTIVE": MCMC_LOCAL_SCAN_OBJECTIVE,
                "PRODUCTION_ESTIMATE_VARIANCE": PRODUCTION_ESTIMATE_VARIANCE,
                "ESTIMATOR_VARIANTS": ["production_only", "scan_plus_production"],
                "MCMC_PROPOSAL_SIZE_BY_SAMPLE_BAND": MCMC_PROPOSAL_SIZE_BY_SAMPLE_BAND,
            },
            indent=2,
        ),
        encoding="utf-8",
    )
    print(f"Saved fixed scan-budget grid outputs to {run_dir}")
else:
    print("SAVE_OUTPUTS is False; nothing saved.")

Saved fixed scan-budget grid outputs to /Users/noamchowers/Documents/University/Thesis/Code/MCMCIS/results/mcmcis_scan_budget_grid/20260408_141712_scan_budget_grid
